# Feature Engineering Prototype
**Source:** `datasets/cleaned_ingested_data/cleaned_ingested_data.csv`  
**Goal:** Engineer all features before Feast materialization into Neon DB  

### Sections
1. Setup & Load Data
2. Target Labels
3. Categorical Encoding
4. Team Form Features (Rolling)
5. Home / Away Split Performance
6. Head-to-Head Features
7. Shot Quality Features
8. Half-Time Pattern Features
9. Referee Features
10. Temporal / Fixture Features
11. Final Feature Set

## 1. Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../datasets/cleaned_ingested_data/cleaned_ingested_data.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min()} â†’ {df['Date'].max()}")
df.head(3)

Shape: (7891, 26)
Date range: 2005-01-10 00:00:00 â†’ 2026-12-02 00:00:00


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HC,AC,HY,AY,HR,AR,season_id,Month,Year,Day
0,2005-01-10,Fulham,Man United,2,3,A,2,3,A,H Webb,...,3,9,2,1,0,0,2005-2006,January,2005,Monday
1,2005-01-10,Portsmouth,Newcastle,0,0,D,0,0,D,S Bennett,...,4,4,1,2,0,0,2005-2006,January,2005,Monday
2,2005-01-10,Sunderland,West Ham,1,1,D,1,0,H,M Atkinson,...,5,3,1,5,0,0,2005-2006,January,2005,Monday


## 2. Target Labels

Pre-compute all prediction targets before any feature engineering.  
These get stored in the feature store alongside the input features.

| Label | Description |
|---|---|
| `result_encoded` | Hâ†’2, Dâ†’1, Aâ†’0 |
| `btts` | Both teams scored (1/0) |
| `over_2_5` | Total goals > 2.5 (1/0) |
| `over_1_5` | Total goals > 1.5 (1/0) |
| `total_goals` | FTHG + FTAG |

In [2]:
df["result_encoded"] = df["FTR"].map({"H": 2, "D": 1, "A": 0})
df["total_goals"]    = df["FTHG"] + df["FTAG"]
df["btts"]           = ((df["FTHG"] > 0) & (df["FTAG"] > 0)).astype(int)
df["over_2_5"]       = (df["total_goals"] > 2).astype(int)
df["over_1_5"]       = (df["total_goals"] > 1).astype(int)

print("Target label distributions:")
print(f"  result_encoded : {df['result_encoded'].value_counts().to_dict()}")
print(f"  btts           : {df['btts'].value_counts().to_dict()}")
print(f"  over_2_5       : {df['over_2_5'].value_counts().to_dict()}")
print(f"  over_1_5       : {df['over_1_5'].value_counts().to_dict()}")
print(f"  total_goals    : mean={df['total_goals'].mean():.2f}, max={df['total_goals'].max()}")

df[["Date", "HomeTeam", "AwayTeam", "FTR", "result_encoded", "total_goals", "btts", "over_2_5", "over_1_5"]].head(5)

Target label distributions:
  result_encoded : {2: 3597, 0: 2381, 1: 1913}
  btts           : {1: 4048, 0: 3843}
  over_2_5       : {1: 4120, 0: 3771}
  over_1_5       : {1: 5993, 0: 1898}
  total_goals    : mean=2.74, max=11


,Date,HomeTeam,AwayTeam,FTR,result_encoded,total_goals,btts,over_2_5,over_1_5
0,2005-01-10,Fulham,Man United,A,0,5,1,1,1
1,2005-01-10,Portsmouth,Newcastle,D,1,0,0,0,0
2,2005-01-10,Sunderland,West Ham,D,1,2,1,0,1
3,2005-01-10,Charlton,Tottenham,A,0,5,1,1,1
4,2005-01-10,Blackburn,West Brom,H,2,2,0,0,1


## 3. Categorical Encoding

Encode all string columns to numeric before feature store materialization.

| Column | Strategy | Reason |
|---|---|---|
| `HTR` | Hâ†’2, Dâ†’1, Aâ†’0 | Same ordinal logic as FTR |
| `HomeTeam` / `AwayTeam` | Label encode | 50+ unique teams, ordinal is fine for tree models |
| `Referee` | Label encode | High cardinality, no natural order |
| `Day` | Ordinal Monâ†’0...Sunâ†’6 | Weekly cycle, natural order |
| `Month` | sin/cos cyclical | Dec and Jan are adjacent â€” raw int breaks this |

In [3]:
from sklearn.preprocessing import LabelEncoder

# --- HTR: same map as FTR ---
df["htr_encoded"] = df["HTR"].map({"H": 2, "D": 1, "A": 0})

# --- Teams & Referee: label encode ---
# Fit on combined home+away so both columns share the same integer space
team_encoder = LabelEncoder()
all_teams = pd.concat([df["HomeTeam"], df["AwayTeam"]]).unique()
team_encoder.fit(all_teams)

df["home_team_encoded"] = team_encoder.transform(df["HomeTeam"])
df["away_team_encoded"] = team_encoder.transform(df["AwayTeam"])

referee_encoder = LabelEncoder()
df["referee_encoded"] = referee_encoder.fit_transform(df["Referee"])

# --- Day: ordinal Mon=0 ... Sun=6 ---
day_order = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
             "Friday": 4, "Saturday": 5, "Sunday": 6}
df["day_encoded"] = df["Day"].map(day_order)

# --- Month: cyclical sin/cos so Dec(12) and Jan(1) stay adjacent ---
month_order = {"January": 1, "February": 2, "March": 3, "April": 4,
               "May": 5, "June": 6, "July": 7, "August": 8,
               "September": 9, "October": 10, "November": 11, "December": 12}
month_num = df["Month"].map(month_order)
df["month_sin"] = np.sin(2 * np.pi * month_num / 12)
df["month_cos"] = np.cos(2 * np.pi * month_num / 12)

# --- Verify ---
encoded_cols = ["htr_encoded", "home_team_encoded", "away_team_encoded",
                "referee_encoded", "day_encoded", "month_sin", "month_cos"]

print(f"Unique teams encoded: {len(team_encoder.classes_)}")
print(f"Unique referees encoded: {len(referee_encoder.classes_)}")
print(f"Nulls in encoded cols: {df[encoded_cols].isnull().sum().to_dict()}")

df[["Date", "HomeTeam", "home_team_encoded", "AwayTeam", "away_team_encoded",
    "HTR", "htr_encoded", "Day", "day_encoded", "Month", "month_sin", "month_cos"]].head(5)

Unique teams encoded: 44
Unique referees encoded: 71
Nulls in encoded cols: {'htr_encoded': 0, 'home_team_encoded': 0, 'away_team_encoded': 0, 'referee_encoded': 0, 'day_encoded': 0, 'month_sin': 0, 'month_cos': 0}


,Date,HomeTeam,home_team_encoded,AwayTeam,away_team_encoded,HTR,htr_encoded,Day,day_encoded,Month,month_sin,month_cos
0,2005-01-10,Fulham,16,Man United,25,A,0,Monday,0,January,0.5,0.866025
1,2005-01-10,Portsmouth,30,Newcastle,27,D,1,Monday,0,January,0.5,0.866025
2,2005-01-10,Sunderland,36,West Ham,41,H,2,Monday,0,January,0.5,0.866025
3,2005-01-10,Charlton,11,Tottenham,38,H,2,Monday,0,January,0.5,0.866025
4,2005-01-10,Blackburn,3,West Brom,40,D,1,Monday,0,January,0.5,0.866025


## 4. Team Form Features (Rolling)

**Strategy:** Convert each match into 2 rows (one per team), compute rolling stats per team sorted by date, then join back.

**Why `shift(1)` matters:**  
Without it, a match's own result is included in its own form feature â€” that's data leakage.  
`shift(1)` pushes each team's stats back one position so the rolling window only sees matches *before* the current one.

| Feature | Window |
|---|---|
| `points_last5/10` | Rolling points (W=3, D=1, L=0) |
| `goals_scored_avg5/10` | Rolling goals scored |
| `goals_conceded_avg5/10` | Rolling goals conceded |
| `sot_avg5` | Rolling shots on target |
| `clean_sheets_last5` | Rolling clean sheet count |
| `win_streak` | Consecutive wins before this match |

In [4]:
# --- Step 4a: Build long-format team match view ---
# Each match becomes 2 rows: one for the home team, one for the away team.
# Stats are always from the team's perspective (goals_scored = goals they scored).

home_view = df[["Date", "HomeTeam", "FTHG", "FTAG", "FTR", "HST"]].copy()
home_view.columns = ["date", "team", "goals_scored", "goals_conceded", "ftr", "sot"]
home_view["points"]      = home_view["ftr"].map({"H": 3, "D": 1, "A": 0})
home_view["won"]         = (home_view["ftr"] == "H").astype(int)
home_view["clean_sheet"] = (home_view["goals_conceded"] == 0).astype(int)

away_view = df[["Date", "AwayTeam", "FTAG", "FTHG", "FTR", "AST"]].copy()
away_view.columns = ["date", "team", "goals_scored", "goals_conceded", "ftr", "sot"]
away_view["points"]      = away_view["ftr"].map({"A": 3, "D": 1, "H": 0})
away_view["won"]         = (away_view["ftr"] == "A").astype(int)
away_view["clean_sheet"] = (away_view["goals_conceded"] == 0).astype(int)

team_matches = (
    pd.concat([home_view, away_view])
    .sort_values(["team", "date"])
    .reset_index(drop=True)
)

print(f"team_matches shape: {team_matches.shape}  (should be {len(df) * 2} rows)")
team_matches[team_matches["team"] == "Liverpool"].head(6)

team_matches shape: (15782, 9)  (should be 15782 rows)


,date,team,goals_scored,goals_conceded,ftr,sot,points,won,clean_sheet
7046,2005-02-10,Liverpool,1,4,A,5,0,0,0
7047,2005-03-12,Liverpool,3,0,H,12,3,1,1
7048,2005-05-11,Liverpool,2,0,A,6,3,1,1
7049,2005-08-13,Liverpool,0,0,D,7,1,0,1
7050,2005-08-20,Liverpool,1,0,H,6,3,1,1
7051,2005-09-18,Liverpool,0,0,D,3,1,0,1


In [5]:
# --- Step 4b: Compute rolling stats with shift(1) for point-in-time correctness ---

roll_cols = ["points", "goals_scored", "goals_conceded", "sot", "clean_sheet", "won"]

def rolling_agg(group, window):
    # shift(1) excludes current match from its own rolling window
    shifted = group[roll_cols].shift(1)
    return shifted.rolling(window, min_periods=1).mean()

form5  = team_matches.groupby("team", group_keys=False).apply(lambda g: rolling_agg(g, 5),  include_groups=False)
form10 = team_matches.groupby("team", group_keys=False).apply(lambda g: rolling_agg(g, 10), include_groups=False)

team_matches["points_last5"]          = form5["points"]
team_matches["goals_scored_avg5"]     = form5["goals_scored"]
team_matches["goals_conceded_avg5"]   = form5["goals_conceded"]
team_matches["sot_avg5"]              = form5["sot"]
team_matches["clean_sheets_last5"]    = form5["clean_sheet"]
team_matches["points_last10"]         = form10["points"]
team_matches["goals_scored_avg10"]    = form10["goals_scored"]
team_matches["goals_conceded_avg10"]  = form10["goals_conceded"]

# --- Win streak: consecutive wins before current match ---
def win_streak(group):
    streaks, count = [], 0
    for won in group["won"].shift(1).fillna(0):
        count = count + 1 if won == 1 else 0
        streaks.append(count)
    return pd.Series(streaks, index=group.index)

team_matches["win_streak"] = team_matches.groupby("team", group_keys=False).apply(win_streak, include_groups=False)

# --- Step 4c: Join form features back to main df ---
form_home = team_matches.rename(columns={
    "team": "HomeTeam", "date": "Date",
    "points_last5": "home_points_last5",
    "goals_scored_avg5": "home_goals_scored_avg5",
    "goals_conceded_avg5": "home_goals_conceded_avg5",
    "sot_avg5": "home_sot_avg5",
    "clean_sheets_last5": "home_clean_sheets_last5",
    "points_last10": "home_points_last10",
    "goals_scored_avg10": "home_goals_scored_avg10",
    "goals_conceded_avg10": "home_goals_conceded_avg10",
    "win_streak": "home_win_streak",
})

form_away = team_matches.rename(columns={
    "team": "AwayTeam", "date": "Date",
    "points_last5": "away_points_last5",
    "goals_scored_avg5": "away_goals_scored_avg5",
    "goals_conceded_avg5": "away_goals_conceded_avg5",
    "sot_avg5": "away_sot_avg5",
    "clean_sheets_last5": "away_clean_sheets_last5",
    "points_last10": "away_points_last10",
    "goals_scored_avg10": "away_goals_scored_avg10",
    "goals_conceded_avg10": "away_goals_conceded_avg10",
    "win_streak": "away_win_streak",
})

home_form_cols = ["Date", "HomeTeam", "home_points_last5", "home_goals_scored_avg5",
                  "home_goals_conceded_avg5", "home_sot_avg5", "home_clean_sheets_last5",
                  "home_points_last10", "home_goals_scored_avg10", "home_goals_conceded_avg10",
                  "home_win_streak"]

away_form_cols = ["Date", "AwayTeam", "away_points_last5", "away_goals_scored_avg5",
                  "away_goals_conceded_avg5", "away_sot_avg5", "away_clean_sheets_last5",
                  "away_points_last10", "away_goals_scored_avg10", "away_goals_conceded_avg10",
                  "away_win_streak"]

df = df.merge(form_home[home_form_cols], on=["Date", "HomeTeam"], how="left")
df = df.merge(form_away[away_form_cols], on=["Date", "AwayTeam"], how="left")

# --- Verify ---
form_feature_cols = [c for c in df.columns if "last5" in c or "last10" in c or "avg5" in c
                     or "avg10" in c or "win_streak" in c]
print(f"Form features added: {len(form_feature_cols)}")
print(f"New df shape: {df.shape}")
print(f"Nulls in form features:\n{df[form_feature_cols].isnull().sum()}")

df[["Date", "HomeTeam", "home_points_last5", "home_goals_scored_avg5",
    "home_win_streak", "AwayTeam", "away_points_last5", "away_goals_conceded_avg5"]].head(8)

Form features added: 18
New df shape: (7891, 56)
Nulls in form features:
home_points_last5            23
home_goals_scored_avg5       23
home_goals_conceded_avg5     23
home_sot_avg5                23
home_clean_sheets_last5      23
home_points_last10           23
home_goals_scored_avg10      23
home_goals_conceded_avg10    23
home_win_streak               0
away_points_last5            21
away_goals_scored_avg5       21
away_goals_conceded_avg5     21
away_sot_avg5                21
away_clean_sheets_last5      21
away_points_last10           21
away_goals_scored_avg10      21
away_goals_conceded_avg10    21
away_win_streak               0
dtype: int64


,Date,HomeTeam,home_points_last5,home_goals_scored_avg5,home_win_streak,AwayTeam,away_points_last5,away_goals_conceded_avg5
0,2005-01-10,Fulham,NaN,NaN,0,Man United,NaN,NaN
1,2005-01-10,Portsmouth,NaN,NaN,0,Newcastle,NaN,NaN
2,2005-01-10,Sunderland,NaN,NaN,0,West Ham,NaN,NaN
3,2005-01-10,Charlton,NaN,NaN,0,Tottenham,NaN,NaN
4,2005-01-10,Blackburn,NaN,NaN,0,West Brom,NaN,NaN
5,2005-02-10,Arsenal,NaN,NaN,0,Birmingham,NaN,NaN
6,2005-02-10,Man City,NaN,NaN,0,Everton,NaN,NaN
7,2005-02-10,Liverpool,NaN,NaN,0,Chelsea,NaN,NaN


## 5. Home / Away Split Performance

Teams perform differently at home vs away. These features capture that split over the last 10 home/away games specifically â€” separate from the general form in Section 4.

In [6]:
# --- Home-specific form: only matches played at home ---
home_only = df[["Date", "HomeTeam", "FTHG", "FTR"]].copy()
home_only.columns = ["date", "team", "goals_scored", "ftr"]
home_only["won"]  = (home_only["ftr"] == "H").astype(int)

home_only = home_only.sort_values(["team", "date"])

def rolling_home(group, window):
    shifted = group[["won", "goals_scored"]].shift(1)
    return shifted.rolling(window, min_periods=1).mean()

home_split = home_only.groupby("team", group_keys=False).apply(
    lambda g: rolling_home(g, 10), include_groups=False
)
home_only["home_win_rate_last10"]   = home_split["won"]
home_only["home_goals_avg_last10"]  = home_split["goals_scored"]

# --- Away-specific form: only matches played away ---
away_only = df[["Date", "AwayTeam", "FTAG", "FTR"]].copy()
away_only.columns = ["date", "team", "goals_scored", "ftr"]
away_only["won"]  = (away_only["ftr"] == "A").astype(int)

away_only = away_only.sort_values(["team", "date"])

def rolling_away(group, window):
    shifted = group[["won", "goals_scored"]].shift(1)
    return shifted.rolling(window, min_periods=1).mean()

away_split = away_only.groupby("team", group_keys=False).apply(
    lambda g: rolling_away(g, 10), include_groups=False
)
away_only["away_win_rate_last10"]   = away_split["won"]
away_only["away_goals_avg_last10"]  = away_split["goals_scored"]

# --- Merge back ---
df = df.merge(
    home_only[["date", "team", "home_win_rate_last10", "home_goals_avg_last10"]]
    .rename(columns={"team": "HomeTeam", "date": "Date"}),
    on=["Date", "HomeTeam"], how="left"
)
df = df.merge(
    away_only[["date", "team", "away_win_rate_last10", "away_goals_avg_last10"]]
    .rename(columns={"team": "AwayTeam", "date": "Date"}),
    on=["Date", "AwayTeam"], how="left"
)

print(f"df shape: {df.shape}")
print(f"Nulls: { {c: df[c].isnull().sum() for c in ['home_win_rate_last10','home_goals_avg_last10','away_win_rate_last10','away_goals_avg_last10']} }")
df[["Date", "HomeTeam", "home_win_rate_last10", "home_goals_avg_last10",
    "AwayTeam", "away_win_rate_last10", "away_goals_avg_last10"]].head(5)

df shape: (7891, 60)
Nulls: {'home_win_rate_last10': np.int64(44), 'home_goals_avg_last10': np.int64(44), 'away_win_rate_last10': np.int64(44), 'away_goals_avg_last10': np.int64(44)}


,Date,HomeTeam,home_win_rate_last10,home_goals_avg_last10,AwayTeam,away_win_rate_last10,away_goals_avg_last10
0,2005-01-10,Fulham,NaN,NaN,Man United,NaN,NaN
1,2005-01-10,Portsmouth,NaN,NaN,Newcastle,NaN,NaN
2,2005-01-10,Sunderland,NaN,NaN,West Ham,NaN,NaN
3,2005-01-10,Charlton,NaN,NaN,Tottenham,NaN,NaN
4,2005-01-10,Blackburn,NaN,NaN,West Brom,NaN,NaN


## 6. Head-to-Head Features

Historical record between these two specific teams before this match date.

In [7]:
# Build a canonical matchup key (alphabetical so Man Utd vs Liverpool == Liverpool vs Man Utd)
h2h_src = df[["Date", "HomeTeam", "AwayTeam", "FTR", "FTHG", "FTAG"]].copy()
h2h_src["team_a"] = h2h_src[["HomeTeam", "AwayTeam"]].min(axis=1)
h2h_src["team_b"] = h2h_src[["HomeTeam", "AwayTeam"]].max(axis=1)
h2h_src["total_goals"] = h2h_src["FTHG"] + h2h_src["FTAG"]
h2h_src["btts_match"]  = ((h2h_src["FTHG"] > 0) & (h2h_src["FTAG"] > 0)).astype(int)
# home_win from HomeTeam perspective (FTR == H)
h2h_src["home_win"]    = (h2h_src["FTR"] == "H").astype(int)
h2h_src = h2h_src.sort_values("Date").reset_index(drop=True)

h2h_features = []
for idx, row in h2h_src.iterrows():
    prior = h2h_src[
        (h2h_src["team_a"] == row["team_a"]) &
        (h2h_src["team_b"] == row["team_b"]) &
        (h2h_src["Date"] < row["Date"])
    ]
    if len(prior) == 0:
        h2h_features.append({"h2h_meetings": 0, "h2h_home_win_rate": np.nan,
                              "h2h_avg_total_goals": np.nan, "h2h_btts_rate": np.nan})
    else:
        h2h_features.append({
            "h2h_meetings":       len(prior),
            "h2h_home_win_rate":  prior["home_win"].mean(),
            "h2h_avg_total_goals":prior["total_goals"].mean(),
            "h2h_btts_rate":      prior["btts_match"].mean(),
        })

h2h_df = pd.DataFrame(h2h_features, index=h2h_src.index)
df = pd.concat([df, h2h_df], axis=1)

print(f"df shape: {df.shape}")
print(f"Nulls: { {c: df[c].isnull().sum() for c in ['h2h_meetings','h2h_home_win_rate','h2h_avg_total_goals','h2h_btts_rate']} }")
df[["Date", "HomeTeam", "AwayTeam", "h2h_meetings", "h2h_home_win_rate",
    "h2h_avg_total_goals", "h2h_btts_rate"]].head(10)

df shape: (7891, 64)
Nulls: {'h2h_meetings': np.int64(0), 'h2h_home_win_rate': np.int64(722), 'h2h_avg_total_goals': np.int64(722), 'h2h_btts_rate': np.int64(722)}


,Date,HomeTeam,AwayTeam,h2h_meetings,h2h_home_win_rate,h2h_avg_total_goals,h2h_btts_rate
0,2005-01-10,Fulham,Man United,0,NaN,NaN,NaN
1,2005-01-10,Portsmouth,Newcastle,0,NaN,NaN,NaN
2,2005-01-10,Sunderland,West Ham,0,NaN,NaN,NaN
3,2005-01-10,Charlton,Tottenham,0,NaN,NaN,NaN
4,2005-01-10,Blackburn,West Brom,0,NaN,NaN,NaN
5,2005-02-10,Arsenal,Birmingham,0,NaN,NaN,NaN
6,2005-02-10,Man City,Everton,0,NaN,NaN,NaN
7,2005-02-10,Liverpool,Chelsea,0,NaN,NaN,NaN
8,2005-02-10,Aston Villa,Middlesbrough,0,NaN,NaN,NaN
9,2005-02-10,Wigan,Bolton,0,NaN,NaN,NaN


## 7. Shot Quality & Efficiency

Rolling shot conversion and on-target ratios â€” how clinical each team has been recently.

In [8]:
# Shot conversion = goals scored / shots on target (rolling avg over last 5)
# On-target ratio  = shots on target / total shots (rolling avg over last 5)

shot_home = df[["Date", "HomeTeam", "FTHG", "HST", "HS"]].copy()
shot_home.columns = ["date", "team", "goals", "sot", "shots"]
shot_home["conversion"] = shot_home["goals"] / shot_home["sot"].replace(0, np.nan)
shot_home["sot_ratio"]  = shot_home["sot"]   / shot_home["shots"].replace(0, np.nan)
shot_home = shot_home.sort_values(["team", "date"])

shot_away = df[["Date", "AwayTeam", "FTAG", "AST", "AS"]].copy()
shot_away.columns = ["date", "team", "goals", "sot", "shots"]
shot_away["conversion"] = shot_away["goals"] / shot_away["sot"].replace(0, np.nan)
shot_away["sot_ratio"]  = shot_away["sot"]   / shot_away["shots"].replace(0, np.nan)
shot_away = shot_away.sort_values(["team", "date"])

def rolling_shot(group):
    shifted = group[["conversion", "sot_ratio"]].shift(1)
    return shifted.rolling(5, min_periods=1).mean()

sh = shot_home.groupby("team", group_keys=False).apply(rolling_shot, include_groups=False)
shot_home["home_conversion_rate_avg5"] = sh["conversion"]
shot_home["home_sot_ratio_avg5"]       = sh["sot_ratio"]

sa = shot_away.groupby("team", group_keys=False).apply(rolling_shot, include_groups=False)
shot_away["away_conversion_rate_avg5"] = sa["conversion"]
shot_away["away_sot_ratio_avg5"]       = sa["sot_ratio"]

df = df.merge(
    shot_home[["date", "team", "home_conversion_rate_avg5", "home_sot_ratio_avg5"]]
    .rename(columns={"team": "HomeTeam", "date": "Date"}),
    on=["Date", "HomeTeam"], how="left"
)
df = df.merge(
    shot_away[["date", "team", "away_conversion_rate_avg5", "away_sot_ratio_avg5"]]
    .rename(columns={"team": "AwayTeam", "date": "Date"}),
    on=["Date", "AwayTeam"], how="left"
)

print(f"df shape: {df.shape}")
df[["Date", "HomeTeam", "home_conversion_rate_avg5", "home_sot_ratio_avg5",
    "AwayTeam", "away_conversion_rate_avg5", "away_sot_ratio_avg5"]].head(5)

df shape: (7891, 68)


,Date,HomeTeam,home_conversion_rate_avg5,home_sot_ratio_avg5,AwayTeam,away_conversion_rate_avg5,away_sot_ratio_avg5
0,2005-01-10,Fulham,NaN,NaN,Man United,NaN,NaN
1,2005-01-10,Portsmouth,NaN,NaN,Newcastle,NaN,NaN
2,2005-01-10,Sunderland,NaN,NaN,West Ham,NaN,NaN
3,2005-01-10,Charlton,NaN,NaN,Tottenham,NaN,NaN
4,2005-01-10,Blackburn,NaN,NaN,West Brom,NaN,NaN


## 8. Half-Time Pattern Features

Second half performance and comeback ability â€” derived from HTHG/HTAG vs FTHG/FTAG.

In [9]:
ht = df[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "HTHG", "HTAG", "HTR", "FTR"]].copy()

# Per-match half-time features (no rolling needed â€” these describe the match pattern)
ht["home_2nd_half_goals"] = ht["FTHG"] - ht["HTHG"]
ht["away_2nd_half_goals"] = ht["FTAG"] - ht["HTAG"]

# Rolling per team: avg second-half goals, comeback rate, lead-hold rate
ht_home = ht[["Date", "HomeTeam", "home_2nd_half_goals", "HTR", "FTR"]].copy()
ht_home.columns = ["date", "team", "sh_goals", "htr", "ftr"]
ht_home["lead_hold"]  = ((ht_home["htr"] == "H") & (ht_home["ftr"] == "H")).astype(int)
ht_home["comeback"]   = ((ht_home["htr"] == "A") & (ht_home["ftr"].isin(["H","D"]))).astype(int)
ht_home = ht_home.sort_values(["team","date"])

ht_away = ht[["Date", "AwayTeam", "away_2nd_half_goals", "HTR", "FTR"]].copy()
ht_away.columns = ["date", "team", "sh_goals", "htr", "ftr"]
ht_away["lead_hold"]  = ((ht_away["htr"] == "A") & (ht_away["ftr"] == "A")).astype(int)
ht_away["comeback"]   = ((ht_away["htr"] == "H") & (ht_away["ftr"].isin(["A","D"]))).astype(int)
ht_away = ht_away.sort_values(["team","date"])

def rolling_ht(group):
    shifted = group[["sh_goals", "lead_hold", "comeback"]].shift(1)
    return shifted.rolling(5, min_periods=1).mean()

rh = ht_home.groupby("team", group_keys=False).apply(rolling_ht, include_groups=False)
ht_home["home_2nd_half_goals_avg5"] = rh["sh_goals"]
ht_home["home_lead_hold_rate"]      = rh["lead_hold"]
ht_home["home_comeback_rate"]       = rh["comeback"]

ra = ht_away.groupby("team", group_keys=False).apply(rolling_ht, include_groups=False)
ht_away["away_2nd_half_goals_avg5"] = ra["sh_goals"]
ht_away["away_lead_hold_rate"]      = ra["lead_hold"]
ht_away["away_comeback_rate"]       = ra["comeback"]

df = df.merge(
    ht_home[["date","team","home_2nd_half_goals_avg5","home_lead_hold_rate","home_comeback_rate"]]
    .rename(columns={"team":"HomeTeam","date":"Date"}),
    on=["Date","HomeTeam"], how="left"
)
df = df.merge(
    ht_away[["date","team","away_2nd_half_goals_avg5","away_lead_hold_rate","away_comeback_rate"]]
    .rename(columns={"team":"AwayTeam","date":"Date"}),
    on=["Date","AwayTeam"], how="left"
)

print(f"df shape: {df.shape}")
df[["Date","HomeTeam","home_2nd_half_goals_avg5","home_lead_hold_rate","home_comeback_rate",
    "AwayTeam","away_2nd_half_goals_avg5","away_lead_hold_rate"]].head(5)

df shape: (7891, 74)


,Date,HomeTeam,home_2nd_half_goals_avg5,home_lead_hold_rate,home_comeback_rate,AwayTeam,away_2nd_half_goals_avg5,away_lead_hold_rate
0,2005-01-10,Fulham,NaN,NaN,NaN,Man United,NaN,NaN
1,2005-01-10,Portsmouth,NaN,NaN,NaN,Newcastle,NaN,NaN
2,2005-01-10,Sunderland,NaN,NaN,NaN,West Ham,NaN,NaN
3,2005-01-10,Charlton,NaN,NaN,NaN,Tottenham,NaN,NaN
4,2005-01-10,Blackburn,NaN,NaN,NaN,West Brom,NaN,NaN


## 9. Referee Features

Referee style aggregated over their full history before each match.

In [10]:
ref_src = df[["Date", "Referee", "HY", "AY", "HF", "AF", "FTR"]].copy()
ref_src["total_yellows"] = ref_src["HY"] + ref_src["AY"]
ref_src["total_fouls"]   = ref_src["HF"] + ref_src["AF"]
ref_src["home_win"]      = (ref_src["FTR"] == "H").astype(int)
ref_src = ref_src.sort_values("Date").reset_index(drop=True)

ref_features = []
for idx, row in ref_src.iterrows():
    prior = ref_src[(ref_src["Referee"] == row["Referee"]) & (ref_src["Date"] < row["Date"])]
    if len(prior) == 0:
        ref_features.append({"ref_avg_yellows": np.nan, "ref_avg_fouls": np.nan,
                              "ref_home_win_rate": np.nan})
    else:
        ref_features.append({
            "ref_avg_yellows":   prior["total_yellows"].mean(),
            "ref_avg_fouls":     prior["total_fouls"].mean(),
            "ref_home_win_rate": prior["home_win"].mean(),
        })

ref_df = pd.DataFrame(ref_features, index=ref_src.index)
df = pd.concat([df, ref_df], axis=1)

print(f"df shape: {df.shape}")
print(f"Nulls: { {c: df[c].isnull().sum() for c in ['ref_avg_yellows','ref_avg_fouls','ref_home_win_rate']} }")
df[["Date","Referee","ref_avg_yellows","ref_avg_fouls","ref_home_win_rate"]].head(8)

df shape: (7891, 77)
Nulls: {'ref_avg_yellows': np.int64(71), 'ref_avg_fouls': np.int64(71), 'ref_home_win_rate': np.int64(71)}


,Date,Referee,ref_avg_yellows,ref_avg_fouls,ref_home_win_rate
0,2005-01-10,H Webb,NaN,NaN,NaN
1,2005-01-10,S Bennett,NaN,NaN,NaN
2,2005-01-10,M Atkinson,NaN,NaN,NaN
3,2005-01-10,P Dowd,NaN,NaN,NaN
4,2005-01-10,U Rennie,NaN,NaN,NaN
5,2005-02-10,C Foy,NaN,NaN,NaN
6,2005-02-10,M Halsey,NaN,NaN,NaN
7,2005-02-10,G Poll,NaN,NaN,NaN


## 10. Temporal / Fixture Features

Match week, season phase, and days rest between matches.

In [11]:
# --- Match week within season (rank of match date per season) ---
df["match_week"] = df.groupby("season_id")["Date"].rank(method="dense").astype(int)

# --- Season phase: Early (1-10), Mid (11-28), Late (29-38) ---
def season_phase(week):
    if week <= 10: return 0   # Early
    elif week <= 28: return 1  # Mid
    else: return 2            # Late

df["season_phase"] = df["match_week"].apply(season_phase)

# --- Days since last match per team (fixture congestion) ---
all_matches = pd.concat([
    df[["Date", "HomeTeam"]].rename(columns={"HomeTeam": "team"}),
    df[["Date", "AwayTeam"]].rename(columns={"AwayTeam": "team"})
]).sort_values(["team", "Date"])

all_matches["days_rest"] = all_matches.groupby("team")["Date"].diff().dt.days

home_rest = all_matches.rename(columns={"team": "HomeTeam", "days_rest": "home_days_rest"})
away_rest = all_matches.rename(columns={"team": "AwayTeam", "days_rest": "away_days_rest"})

df = df.merge(home_rest[["Date", "HomeTeam", "home_days_rest"]], on=["Date", "HomeTeam"], how="left")
df = df.merge(away_rest[["Date", "AwayTeam", "away_days_rest"]], on=["Date", "AwayTeam"], how="left")

# First match of each team has no prior match — fill with season median
median_rest = df["home_days_rest"].median()
df["home_days_rest"] = df["home_days_rest"].fillna(median_rest)
df["away_days_rest"] = df["away_days_rest"].fillna(median_rest)

print(f"df shape: {df.shape}")
print(f"match_week range: {df['match_week'].min()} - {df['match_week'].max()}")
print(f"season_phase counts: {df['season_phase'].value_counts().to_dict()}")
df[["Date", "season_id", "match_week", "season_phase", "HomeTeam", "home_days_rest",
    "AwayTeam", "away_days_rest"]].head(8)

df shape: (7891, 81)
match_week range: 1 - 135
season_phase counts: {2: 5677, 1: 1416, 0: 798}


,Date,season_id,match_week,season_phase,HomeTeam,home_days_rest,AwayTeam,away_days_rest
0,2005-01-10,2005-2006,1,0,Fulham,7.0,Man United,7.0
1,2005-01-10,2005-2006,1,0,Portsmouth,7.0,Newcastle,7.0
2,2005-01-10,2005-2006,1,0,Sunderland,7.0,West Ham,7.0
3,2005-01-10,2005-2006,1,0,Charlton,7.0,Tottenham,7.0
4,2005-01-10,2005-2006,1,0,Blackburn,7.0,West Brom,7.0
5,2005-02-10,2005-2006,2,0,Arsenal,7.0,Birmingham,7.0
6,2005-02-10,2005-2006,2,0,Man City,7.0,Everton,7.0
7,2005-02-10,2005-2006,2,0,Liverpool,7.0,Chelsea,7.0


## 11. Final Feature Set

Fill remaining NaNs, select engineered feature columns, print final shape and save to .

In [13]:
# --- Fill NaNs ---
# Rolling features for earliest matches have NaN (no prior history) → fill with 0
# H2H and referee NaNs for first encounters → fill with dataset-level means

h2h_cols = ["h2h_home_win_rate", "h2h_avg_total_goals", "h2h_btts_rate"]
ref_cols  = ["ref_avg_yellows", "ref_avg_fouls", "ref_home_win_rate"]

for col in h2h_cols + ref_cols:
    df[col] = df[col].fillna(df[col].mean())

rolling_like = [c for c in df.columns if any(x in c for x in
    ["points_last", "goals_scored", "goals_conceded", "sot_avg", "clean_sheets",
     "win_rate", "goals_avg", "conversion_rate", "sot_ratio",
     "2nd_half_goals", "lead_hold", "comeback"])]
df[rolling_like] = df[rolling_like].fillna(0)

# --- Define final feature columns (input to model) ---
target_cols   = ["result_encoded", "btts", "over_2_5", "over_1_5", "total_goals"]
id_cols       = ["Date", "HomeTeam", "AwayTeam", "season_id"]
raw_cols      = [c for c in df.columns[:26]]  # original 26 raw columns

feature_cols = [c for c in df.columns if c not in raw_cols and c not in target_cols]

print(f"Final df shape       : {df.shape}")
print(f"Engineered features  : {len(feature_cols)}")
print(f"Target labels        : {len(target_cols)}")
print(f"Remaining NaNs       : {df[feature_cols].isnull().sum().sum()}")
print()
print("Feature columns:")
for c in feature_cols: print(f"  {c}")

# --- Save to datasets/processed/ ---
OUT_DIR = Path("../../datasets/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / "feature_engineered_dataset.csv"
df.to_csv(out_path, index=False)
print(f"Saved to: {out_path}")

Final df shape       : (7891, 81)
Engineered features  : 50
Target labels        : 5
Remaining NaNs       : 0

Feature columns:
  htr_encoded
  home_team_encoded
  away_team_encoded
  referee_encoded
  day_encoded
  month_sin
  month_cos
  home_points_last5
  home_goals_scored_avg5
  home_goals_conceded_avg5
  home_sot_avg5
  home_clean_sheets_last5
  home_points_last10
  home_goals_scored_avg10
  home_goals_conceded_avg10
  home_win_streak
  away_points_last5
  away_goals_scored_avg5
  away_goals_conceded_avg5
  away_sot_avg5
  away_clean_sheets_last5
  away_points_last10
  away_goals_scored_avg10
  away_goals_conceded_avg10
  away_win_streak
  home_win_rate_last10
  home_goals_avg_last10
  away_win_rate_last10
  away_goals_avg_last10
  h2h_meetings
  h2h_home_win_rate
  h2h_avg_total_goals
  h2h_btts_rate
  home_conversion_rate_avg5
  home_sot_ratio_avg5
  away_conversion_rate_avg5
  away_sot_ratio_avg5
  home_2nd_half_goals_avg5
  home_lead_hold_rate
  home_comeback_rate
  away_2nd_